# Merge bioschemas `$validation` blocks

Merges two draft bioschemas profile JSON files into one:

- **Cardinality / marginality source** — supplies `description`, `owl:cardinality`,
  and the top-level `required` / `recommended` / `optional` lists for each property.
- **JSON-Schema rules source** — supplies the actual JSON-Schema validation keywords
  (`anyOf`, `oneOf`, `type`, `items`, `format`, etc.) for each property.

**Run this notebook's kernel under the `bioschemas` conda environment**
(`Anaconda3\envs\bioschemas`). Only the standard library is used (`json`, `pathlib`,
`copy`) so no extra installs are needed in that env.

> This produces a **draft** to speed up a curator's review, not an authoritative
> auto-merge. Anything ambiguous is flagged in the printed output rather than
> silently guessed.

**There is a mandatory human-confirmation stop partway through this notebook.**
Do not run past the cell marked `STOP - human confirmation gate` until a person has
reviewed the single-property test merge and confirmed it looks correct.

## 1. Parameters

Edit these if filenames or namespaces change.

In [ ]:
import json
import copy
from pathlib import Path

ROOT_DIR = Path(r"C:\Users\gtsueng\Anaconda3\envs\bioschemas\schema_validation_merger")
INPUT_DIR = ROOT_DIR / "files" / "to merge"
OUTPUT_DIR = ROOT_DIR / "output"

# Default filenames -- used as a hint, but roles are cross-checked against the
# actual contents of $validation (see detect_roles below), so this keeps working
# even if a future pair of files is named differently.
cardinality_source_file = "Nick-Sample-Profile.json"
schema_rules_source_file = "Ginger-Sample-Profile.json"

# Namespace prefixes to normalize -> target namespace.
NAMESPACE_REPLACEMENTS = {
    "NicktempSample": "bioschemasdrafts",
    "gtest": "bioschemasdrafts",
}

# Canonical URI the renamed prefix should point at in @context (follows the same
# pattern as the existing bioschemastypesdrafts namespace already in these files).
TARGET_NAMESPACE_URI = "https://discovery.biothings.io/ns/bioschemasdrafts/"

OUTPUT_FILENAME = "merged-Sample-Profile.json"

## 2. Load input files and detect roles

Loads every `*.json` file actually present in `files/to merge/`, then figures out
which one is the cardinality/marginality source and which is the JSON-Schema-rules
source -- by filename first, cross-checked against which file's top-level
`$validation.required/recommended/optional` lists are actually populated (the
rules-only file typically ships those as empty placeholder lists).

In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def first_graph_validation(doc):
    """Return (graph_entry, validation_dict) for the first @graph item that has $validation."""
    for entry in doc.get("@graph", []):
        if "$validation" in entry:
            return entry, entry["$validation"]
    raise ValueError("No @graph entry with a $validation block was found.")


def has_marginality(validation):
    return any(validation.get(key) for key in ("required", "recommended", "optional"))


input_files = sorted(INPUT_DIR.glob("*.json"))
if not input_files:
    raise FileNotFoundError(f"No JSON files found in {INPUT_DIR}")

print("Files found in files/to merge/:")
for f in input_files:
    print(" -", f.name)

In [ ]:
def normalize(name: str) -> str:
    return name.lower().replace("_", "-")


by_name = {normalize(f.name): f for f in input_files}

cardinality_path = by_name.get(normalize(cardinality_source_file))
schema_rules_path = by_name.get(normalize(schema_rules_source_file))

if cardinality_path is None or schema_rules_path is None or cardinality_path == schema_rules_path:
    # Fall back to content-based auto-detection across whatever files are present.
    print("Configured filenames not both found -- auto-detecting roles from content instead.")
    candidates = []
    for f in input_files:
        doc = load_json(f)
        _, validation = first_graph_validation(doc)
        candidates.append((f, doc, has_marginality(validation)))
    marginality_files = [f for f, _, has_marg in candidates if has_marg]
    if len(marginality_files) == 1:
        cardinality_path = marginality_files[0]
        schema_rules_path = next(f for f, _, has_marg in candidates if not has_marg)
    else:
        raise ValueError(
            "Could not auto-detect roles -- more than one (or zero) input files have "
            "populated required/recommended/optional lists. Set cardinality_source_file "
            "and schema_rules_source_file explicitly in the Parameters cell."
        )

cardinality_doc = load_json(cardinality_path)
schema_rules_doc = load_json(schema_rules_path)

cardinality_entry, cardinality_validation = first_graph_validation(cardinality_doc)
schema_entry, schema_validation = first_graph_validation(schema_rules_doc)

# Sanity cross-check: warn (don't fail) if the file we picked as the cardinality
# source doesn't actually look like it has marginality info, or vice versa.
if not has_marginality(cardinality_validation):
    print(f"WARNING: cardinality source '{cardinality_path.name}' has no populated "
          f"required/recommended/optional lists -- double check role assignment.")
if has_marginality(schema_validation):
    print(f"NOTE: schema-rules source '{schema_rules_path.name}' also has populated "
          f"required/recommended/optional lists; these will be IGNORED per the merge spec "
          f"(marginality always comes from the cardinality source).")

print(f"Cardinality / marginality source: {cardinality_path.name}")
print(f"JSON-Schema rules source:         {schema_rules_path.name}")

## 3. Extract the two pieces of information from each source

In [ ]:
def extract_cardinality_info(validation):
    """property name -> {description?, owl:cardinality?}, plus the marginality lists."""
    props = {}
    for name, body in validation.get("properties", {}).items():
        info = {}
        if "description" in body:
            info["description"] = body["description"]
        if "owl:cardinality" in body:
            info["owl:cardinality"] = body["owl:cardinality"]
        props[name] = info
    return props


def extract_schema_rules(validation):
    """property name -> full JSON-Schema keyword dict, with that file's own description dropped."""
    props = {}
    for name, body in validation.get("properties", {}).items():
        props[name] = {k: copy.deepcopy(v) for k, v in body.items() if k != "description"}
    return props


cardinality_props = extract_cardinality_info(cardinality_validation)
schema_props = extract_schema_rules(schema_validation)

print(f"{len(cardinality_props)} properties in cardinality source, "
      f"{len(schema_props)} properties in schema-rules source.")

## 4. Merge logic (single property)

In [ ]:
def merge_property(cardinality_info, schema_info):
    """cardinality_info / schema_info may each be None if the property is missing on that side."""
    merged = {}
    if cardinality_info:
        if "description" in cardinality_info:
            merged["description"] = cardinality_info["description"]
        if "owl:cardinality" in cardinality_info:
            merged["owl:cardinality"] = cardinality_info["owl:cardinality"]
    if schema_info:
        for k, v in schema_info.items():
            merged[k] = copy.deepcopy(v)
    return merged

## 5. Test merge -- ONE property only

Pick a single property common to both sources and merge just that one, so a human
can sanity-check the merge logic before it's applied to everything.

In [ ]:
common_properties = sorted(set(cardinality_props) & set(schema_props))
if not common_properties:
    raise ValueError("No property name is common to both sources -- cannot build a test merge.")

test_property = common_properties[0]
test_merge_result = merge_property(cardinality_props[test_property], schema_props[test_property])

print(f"Test merge for property: {test_property!r}\n")
print(json.dumps(test_merge_result, indent=2))

## ⏸ STOP - human confirmation gate

**Do not run any cell below this one yet.**

Look at the test merge printed just above and confirm:

- the `description` is the one from the cardinality source (not duplicated/overwritten
  by the schema-rules file's own description),
- `owl:cardinality` is present and correct,
- the JSON-Schema keywords (`anyOf` / `oneOf` / `type` / `items` / ...) are the ones
  from the schema-rules source, unmodified,
- nothing unexpected got merged in.

If you are Claude running this on the user's behalf: **stop here, show the printed
test merge to the user in the chat, and explicitly ask them to confirm it before you
run the remaining cells.** Do not decide on your own that it looks fine.

If something is wrong, fix the logic in the cells above, rerun the test merge, and
get confirmation again before proceeding.

## 6. Full merge -- all properties

Only run this after the test merge above has been confirmed. Merges the union of
property names from both sources and flags anything missing from one side instead
of silently dropping or inventing data.

In [ ]:
all_property_names = sorted(set(cardinality_props) | set(schema_props))

missing_from_cardinality = [n for n in all_property_names if n not in cardinality_props]
missing_from_schema = [n for n in all_property_names if n not in schema_props]

merged_properties = {
    name: merge_property(cardinality_props.get(name), schema_props.get(name))
    for name in all_property_names
}

if missing_from_cardinality:
    print("FLAGGED -- no description/owl:cardinality found (only has JSON-Schema rules):")
    for n in missing_from_cardinality:
        print("  -", n)
if missing_from_schema:
    print("FLAGGED -- no JSON-Schema rules found (only has description/owl:cardinality):")
    for n in missing_from_schema:
        print("  -", n)
if not missing_from_cardinality and not missing_from_schema:
    print("Every property was present in both sources -- nothing flagged.")

In [ ]:
# Marginality lists come only from the cardinality source, filtered down to
# properties that actually made it into the merged set.
merged_required = [n for n in cardinality_validation.get("required", []) if n in merged_properties]
merged_recommended = [n for n in cardinality_validation.get("recommended", []) if n in merged_properties]
merged_optional = [n for n in cardinality_validation.get("optional", []) if n in merged_properties]

merged_validation = {
    "$schema": schema_validation.get("$schema", cardinality_validation.get("$schema")),
    "type": "object",
    "properties": merged_properties,
    "required": merged_required,
    "recommended": merged_recommended,
    "optional": merged_optional,
}

print(f"Merged $validation now has {len(merged_properties)} properties.")

## 7. Namespace normalization

Replaces every configured namespace prefix (default `NicktempSample`, `gtest` ->
`bioschemasdrafts`) wherever it appears -- as an `@context` key, inside a CURIE
string like `"NicktempSample:Sample"` or `"@type": "gtest:Foo"`, or anywhere nested
inside `$validation` -- rather than only touching the top-level `@context`/`@id`,
since future input files might reference these namespaces from inside a property's
validation rules too.

In [ ]:
def replace_namespaces(obj, mapping):
    """Recursively rewrite CURIE-style 'prefix:rest' strings and dict keys using mapping."""
    if isinstance(obj, dict):
        new_obj = {}
        for key, value in obj.items():
            new_key = mapping.get(key, key)
            new_obj[new_key] = replace_namespaces(value, mapping)
        return new_obj
    if isinstance(obj, list):
        return [replace_namespaces(item, mapping) for item in obj]
    if isinstance(obj, str) and ":" in obj:
        prefix, _, rest = obj.partition(":")
        if prefix in mapping:
            return f"{mapping[prefix]}:{rest}"
    return obj

## 8. Assemble the merged document and write output

In [ ]:
# Base the output document on the cardinality source's overall profile (context,
# @id, rdfs:comment/label, schemaVersion, etc.), then drop in the merged $validation
# and normalize namespaces across the whole thing.
merged_document = copy.deepcopy(cardinality_doc)
merged_entry, _ = first_graph_validation(merged_document)
merged_entry["$validation"] = merged_validation

merged_document = replace_namespaces(merged_document, NAMESPACE_REPLACEMENTS)

# @context *values* are namespace URIs, not CURIEs, so the generic CURIE-based
# replacement above intentionally leaves them untouched -- fix the renamed prefix's
# URI explicitly so it points at the canonical namespace instead of an old draft's URL.
context = merged_document.get("@context", {})
for new_prefix in set(NAMESPACE_REPLACEMENTS.values()):
    if new_prefix in context:
        context[new_prefix] = TARGET_NAMESPACE_URI

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
output_path = OUTPUT_DIR / OUTPUT_FILENAME
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(merged_document, f, indent=2, ensure_ascii=False)
    f.write("\n")

print(f"Wrote merged profile to: {output_path}")